# Occuspace preprocessing — Step 1: combine exports

Combine three 30-minute Occuspace exports and strip file-level metadata so each file is a single tidy table.

**Source files**
- `-30minExport-1May23-1May24.csv`
- `-30minExport-1Jun24-1Jun25.csv`
- `-30minExport-1Jun25-15Apr26.csv`

Each export has the same layout:
- **Rows 1–5:** metadata (`Customer`, `Date Range`, `Hour Range`, `Days of the Week`, `Time Interval`)
- **Row 6:** blank
- **Row 7:** column names
- **Row 8+:** data

In [1]:
from pathlib import Path

import pandas as pd

BASE = Path.cwd()
if not (BASE / "-30minExport-1Jun24-1Jun25.csv").exists():
    BASE = Path.cwd() / "ClassProject"

PROCESSED_DIR = BASE / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

FILES = [
    ("2023-05_to_2024-05", "-30minExport-1May23-1May24.csv"),
    ("2024-06_to_2025-06", "-30minExport-1Jun24-1Jun25.csv"),
    ("2025-06_to_2026-04", "-30minExport-1Jun25-15Apr26.csv"),
]

COMBINED_PATH = PROCESSED_DIR / "occuspace_30min_combined.csv"
METADATA_ROWS = 5  # Occuspace metadata lines before the blank row
SKIPROWS = METADATA_ROWS + 1  # metadata + blank line; row 7 is the header

BASE

PosixPath('/Users/pmazolew/Documents/GitHub/AdvancedMachineLearning/ClassProject')

## 1. Read and record export metadata (top 5 rows)

These rows are not part of the occupancy table; we parse them for provenance and then drop them when loading data.

In [2]:
def read_export_metadata(path: Path) -> dict[str, str]:
    """Parse the first METADATA_ROWS key,value lines from an Occuspace export."""
    meta: dict[str, str] = {}
    with path.open(encoding="utf-8") as f:
        for _ in range(METADATA_ROWS):
            line = f.readline().rstrip("\n")
            if "," in line:
                key, value = line.split(",", 1)
                meta[key.strip()] = value.strip()
    return meta


metadata_records = []
for period, filename in FILES:
    path = BASE / filename
    meta = read_export_metadata(path)
    meta["export_period"] = period
    meta["source_file"] = filename
    metadata_records.append(meta)

metadata_df = pd.DataFrame(metadata_records)
metadata_df

,Customer,Date Range,Hour Range,Days of the Week,Time Interval,export_period,source_file
0,Occuspace,05/01/2023 - 05/01/2024,6 - 23,Sun - Sat,30min,2023-05_to_2024-05,-30minExport-1May23-1May24.csv
1,Occuspace,06/01/2024 - 06/01/2025,6 - 23,Sun - Sat,30min,2024-06_to_2025-06,-30minExport-1Jun24-1Jun25.csv
2,Occuspace,06/01/2025 - 04/15/2026,6 - 23,Sun - Sat,30min,2025-06_to_2026-04,-30minExport-1Jun25-15Apr26.csv


In [3]:
metadata_df.to_csv(PROCESSED_DIR / "export_metadata.csv", index=False)

## 2. Load data (skip metadata + blank row) and combine

`skiprows=6` removes the five metadata lines and the blank line; pandas uses the next line as the column header.

In [4]:
def load_export(period: str, filename: str) -> pd.DataFrame:
    path = BASE / filename
    df = pd.read_csv(path, skiprows=SKIPROWS)
    df["export_period"] = period
    df["source_file"] = filename
    return df


frames = [load_export(period, filename) for period, filename in FILES]
df = pd.concat(frames, ignore_index=True)

print(f"Combined shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nRows per export:")
print(df["export_period"].value_counts().sort_index().to_string())
df.head()

Combined shape: 223,499 rows × 15 columns

Rows per export:
export_period
2023-05_to_2024-05    76199
2024-06_to_2025-06    78815
2025-06_to_2026-04    68485


,Location,Timestamp,Date,Day of Week,Week of Year,Time,Hour of Day,Average Occupancy,Average Utilization,Peak Occupancy,Peak Utilization,Capacity,Location Path,export_period,source_file
0,Lower Exercise Room,05/13/2023 6:00,5/13/2023,7,19,6:00:00 AM,6,2,0.02,3,0.04,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...,2023-05_to_2024-05,-30minExport-1May23-1May24.csv
1,Lower Exercise Room,05/13/2023 6:30,5/13/2023,7,19,6:30:00 AM,6,2,0.02,3,0.04,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...,2023-05_to_2024-05,-30minExport-1May23-1May24.csv
2,Lower Exercise Room,05/13/2023 7:00,5/13/2023,7,19,7:00:00 AM,7,2,0.02,3,0.04,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...,2023-05_to_2024-05,-30minExport-1May23-1May24.csv
3,Lower Exercise Room,05/13/2023 7:30,5/13/2023,7,19,7:30:00 AM,7,3,0.04,3,0.04,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...,2023-05_to_2024-05,-30minExport-1May23-1May24.csv
4,Lower Exercise Room,05/13/2023 8:00,5/13/2023,7,19,8:00:00 AM,8,10,0.12,15,0.18,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...,2023-05_to_2024-05,-30minExport-1May23-1May24.csv


## 3. Save combined table (unfiltered)

In [5]:
df.to_csv(COMBINED_PATH, index=False)
print(f"Wrote {COMBINED_PATH}")

Wrote /Users/pmazolew/Documents/GitHub/AdvancedMachineLearning/ClassProject/processed/occuspace_30min_combined.csv


## 4. Filter to academic quarters (weeks 1–10 + finals)

Keep rows only during **instruction** (classes begin through last day of classes, weeks 1–10) and **finals week**, using the Cal Poly academic calendar. Everything else—summer, winter break, spring break, and gaps between terms—is dropped.

Dates are taken from the university calendar for 2022–23 through 2025–26. Spring 2023 is included for the partial export that starts in May 2023.

**Note:** Occuspace exports do not cover every calendar day (e.g. the first file ends 2025-05-01 and the next starts 2025-06-01, so May 2024 is absent). The filter keeps all rows that fall in a term window; missing days are gaps in the source exports, not the filter.

In [6]:
# Cal Poly term windows: instruction (wk 1–10) + finals
# Columns: academic_year, term, classes_begin, last_day_classes, finals_begin, finals_end
ACADEMIC_TERMS = pd.DataFrame(
    [
        # 2022–23 (partial — export begins May 2023)
        ("2022-23", "Spring", "2023-04-03", "2023-06-09", "2023-06-12", "2023-06-16"),
        # 2023–24
        ("2023-24", "Fall", "2023-09-21", "2023-12-08", "2023-12-11", "2023-12-15"),
        ("2023-24", "Winter", "2024-01-08", "2024-03-15", "2024-03-18", "2024-03-22"),
        ("2023-24", "Spring", "2024-04-02", "2024-06-07", "2024-06-10", "2024-06-14"),
        # 2024–25
        ("2024-25", "Fall", "2024-09-23", "2024-12-06", "2024-12-09", "2024-12-13"),
        ("2024-25", "Winter", "2025-01-06", "2025-03-14", "2025-03-17", "2025-03-21"),
        ("2024-25", "Spring", "2025-04-01", "2025-06-06", "2025-06-09", "2025-06-13"),
        # 2025–26
        ("2025-26", "Fall", "2025-09-18", "2025-12-05", "2025-12-08", "2025-12-12"),
        ("2025-26", "Winter", "2026-01-05", "2026-03-13", "2026-03-16", "2026-03-20"),
        ("2025-26", "Spring", "2026-03-30", "2026-06-05", "2026-06-08", "2026-06-12"),
    ],
    columns=[
        "academic_year",
        "term",
        "classes_begin",
        "last_day_classes",
        "finals_begin",
        "finals_end",
    ],
)

date_cols = ["classes_begin", "last_day_classes", "finals_begin", "finals_end"]
ACADEMIC_TERMS[date_cols] = ACADEMIC_TERMS[date_cols].apply(pd.to_datetime)

ACADEMIC_TERMS.to_csv(PROCESSED_DIR / "academic_terms.csv", index=False)
ACADEMIC_TERMS

,academic_year,term,classes_begin,last_day_classes,finals_begin,finals_end
0,2022-23,Spring,2023-04-03,2023-06-09,2023-06-12,2023-06-16
1,2023-24,Fall,2023-09-21,2023-12-08,2023-12-11,2023-12-15
2,2023-24,Winter,2024-01-08,2024-03-15,2024-03-18,2024-03-22
3,2023-24,Spring,2024-04-02,2024-06-07,2024-06-10,2024-06-14
4,2024-25,Fall,2024-09-23,2024-12-06,2024-12-09,2024-12-13
5,2024-25,Winter,2025-01-06,2025-03-14,2025-03-17,2025-03-21
6,2024-25,Spring,2025-04-01,2025-06-06,2025-06-09,2025-06-13
7,2025-26,Fall,2025-09-18,2025-12-05,2025-12-08,2025-12-12
8,2025-26,Winter,2026-01-05,2026-03-13,2026-03-16,2026-03-20
9,2025-26,Spring,2026-03-30,2026-06-05,2026-06-08,2026-06-12


In [7]:
def assign_academic_term(dates: pd.Series) -> pd.DataFrame:
    """Map each date to academic_year, term, and phase (instruction | finals)."""
    out = pd.DataFrame(index=dates.index)
    out["academic_year"] = pd.NA
    out["term"] = pd.NA
    out["term_phase"] = pd.NA

    for _, row in ACADEMIC_TERMS.iterrows():
        instruction = dates.between(row.classes_begin, row.last_day_classes)
        finals = dates.between(row.finals_begin, row.finals_end)
        in_term = instruction | finals

        out.loc[in_term, "academic_year"] = row.academic_year
        out.loc[in_term, "term"] = row.term
        out.loc[instruction, "term_phase"] = "instruction"
        out.loc[finals, "term_phase"] = "finals"

    return out


df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%m/%d/%Y %H:%M")
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")

term_cols = assign_academic_term(df["Date"])
df = pd.concat([df, term_cols], axis=1)

# Instruction week: 1–10 during instruction; 11 during finals
df["instruction_week"] = pd.NA
for _, row in ACADEMIC_TERMS.iterrows():
    term_mask = (df["academic_year"] == row.academic_year) & (df["term"] == row.term)
    instr_mask = term_mask & (df["term_phase"] == "instruction")
    days = (df.loc[instr_mask, "Date"] - row.classes_begin).dt.days
    df.loc[instr_mask, "instruction_week"] = (days // 7 + 1).clip(upper=10)
    df.loc[term_mask & (df["term_phase"] == "finals"), "instruction_week"] = 11

rows_before = len(df)
df_terms = df.dropna(subset=["academic_year"]).copy()
rows_after = len(df_terms)

print(f"Rows before filter: {rows_before:,}")
print(f"Rows in academic terms: {rows_after:,} ({100 * rows_after / rows_before:.1f}%)")
print(f"Rows dropped: {rows_before - rows_after:,}")
print("\nRows per term:")
print(
    df_terms.groupby(["academic_year", "term"], observed=True)
    .size()
    .rename("rows")
    .to_string()
)
df_terms.head()

Rows before filter: 223,499
Rows in academic terms: 136,365 (61.0%)
Rows dropped: 87,134

Rows per term:
academic_year  term  
2022-23        Spring     7128
2023-24        Fall      18144
               Spring     9072
               Winter    15768
2024-25        Fall      17228
               Spring    15762
               Winter    15719
2025-26        Fall      18115
               Spring     3668
               Winter    15761


,Location,Timestamp,Date,Day of Week,Week of Year,Time,Hour of Day,Average Occupancy,Average Utilization,Peak Occupancy,Peak Utilization,Capacity,Location Path,export_period,source_file,academic_year,term,term_phase,instruction_week
0,Lower Exercise Room,2023-05-13 06:00:00,2023-05-13,7,19,6:00:00 AM,6,2,0.02,3,0.04,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...,2023-05_to_2024-05,-30minExport-1May23-1May24.csv,2022-23,Spring,instruction,6
1,Lower Exercise Room,2023-05-13 06:30:00,2023-05-13,7,19,6:30:00 AM,6,2,0.02,3,0.04,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...,2023-05_to_2024-05,-30minExport-1May23-1May24.csv,2022-23,Spring,instruction,6
2,Lower Exercise Room,2023-05-13 07:00:00,2023-05-13,7,19,7:00:00 AM,7,2,0.02,3,0.04,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...,2023-05_to_2024-05,-30minExport-1May23-1May24.csv,2022-23,Spring,instruction,6
3,Lower Exercise Room,2023-05-13 07:30:00,2023-05-13,7,19,7:30:00 AM,7,3,0.04,3,0.04,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...,2023-05_to_2024-05,-30minExport-1May23-1May24.csv,2022-23,Spring,instruction,6
4,Lower Exercise Room,2023-05-13 08:00:00,2023-05-13,7,19,8:00:00 AM,8,10,0.12,15,0.18,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...,2023-05_to_2024-05,-30minExport-1May23-1May24.csv,2022-23,Spring,instruction,6


In [8]:
ACADEMIC_PATH = PROCESSED_DIR / "occuspace_30min_academic_terms.csv"

df_terms.to_csv(ACADEMIC_PATH, index=False)
print(f"Wrote {ACADEMIC_PATH}")

Wrote /Users/pmazolew/Documents/GitHub/AdvancedMachineLearning/ClassProject/processed/occuspace_30min_academic_terms.csv


## 5. Filter to Rec Center (building total)

Keep only the parent **`Rec Center`** location (capacity 220). This is the building-level aggregate, equivalent to the sum of the three exercise rooms. Floor- and room-level rows are dropped to avoid double counting.

In [9]:
REC_CENTER = "Rec Center"

df_rec = df_terms[df_terms["Location"] == REC_CENTER].copy()
REC_CENTER_PATH = PROCESSED_DIR / "occuspace_30min_rec_center.csv"

print(f"Rows before: {len(df_terms):,}")
print(f"Rows after (Rec Center only): {len(df_rec):,}")
print(f"Capacity: {df_rec['Capacity'].iloc[0]}")
print(f"Date range: {df_rec['Date'].min().date()} → {df_rec['Date'].max().date()}")

df_rec.to_csv(REC_CENTER_PATH, index=False)
print(f"\nWrote {REC_CENTER_PATH}")
df_rec.head()

Rows before: 136,365
Rows after (Rec Center only): 22,744
Capacity: 220
Date range: 2023-05-13 → 2026-04-15

Wrote /Users/pmazolew/Documents/GitHub/AdvancedMachineLearning/ClassProject/processed/occuspace_30min_rec_center.csv


,Location,Timestamp,Date,Day of Week,Week of Year,Time,Hour of Day,Average Occupancy,Average Utilization,Peak Occupancy,Peak Utilization,Capacity,Location Path,export_period,source_file,academic_year,term,term_phase,instruction_week
63485,Rec Center,2023-05-13 06:00:00,2023-05-13,7,19,6:00:00 AM,6,5,0.02,6,0.03,220,Cal Poly > Rec Center,2023-05_to_2024-05,-30minExport-1May23-1May24.csv,2022-23,Spring,instruction,6
63486,Rec Center,2023-05-13 06:30:00,2023-05-13,7,19,6:30:00 AM,6,5,0.02,6,0.03,220,Cal Poly > Rec Center,2023-05_to_2024-05,-30minExport-1May23-1May24.csv,2022-23,Spring,instruction,6
63487,Rec Center,2023-05-13 07:00:00,2023-05-13,7,19,7:00:00 AM,7,5,0.02,6,0.03,220,Cal Poly > Rec Center,2023-05_to_2024-05,-30minExport-1May23-1May24.csv,2022-23,Spring,instruction,6
63488,Rec Center,2023-05-13 07:30:00,2023-05-13,7,19,7:30:00 AM,7,7,0.03,8,0.04,220,Cal Poly > Rec Center,2023-05_to_2024-05,-30minExport-1May23-1May24.csv,2022-23,Spring,instruction,6
63489,Rec Center,2023-05-13 08:00:00,2023-05-13,7,19,8:00:00 AM,8,48,0.22,64,0.29,220,Cal Poly > Rec Center,2023-05_to_2024-05,-30minExport-1May23-1May24.csv,2022-23,Spring,instruction,6
